Generating the dependencies for the EDA

In [53]:
import pandas as pd
import numpy as np
data=pd.read_csv(r'C:\Users\Venkat\Desktop\IIITB\flip hackathon\Traffic-demand-prediction\dataset\train.csv')
test_data=pd.read_csv(r'C:\Users\Venkat\Desktop\IIITB\flip hackathon\Traffic-demand-prediction\dataset\test.csv')
print('Data loaded successfully!')


Data loaded successfully!


In [54]:
print(data.shape)
print(test_data.shape)

(77299, 11)
(41778, 10)


In [55]:
print(data.head())#to have an idea about how the columns actually look like

   Index geohash  day timestamp    demand     RoadType  NumberofLanes  \
0      0  qp02z1   48       0:0  0.048804          NaN              1   
1      1  qp02zt   48       0:0  0.118507  Residential              3   
2      2  qp08bj   48       0:0  0.027132  Residential              1   
3      3  qp08gt   48       0:0  0.003272  Residential              1   
4      4  qp02zq   48       0:0  0.010819  Residential              1   

  LargeVehicles Landmarks  Temperature Weather  
0   Not Allowed        No          NaN     NaN  
1       Allowed       Yes    31.104565   Sunny  
2   Not Allowed        No    25.919267   Sunny  
3   Not Allowed        No          NaN   Rainy  
4   Not Allowed        No    10.803667   Rainy  


Here we can see that we have some missing data, let us see how much of our data is actually missing

In [56]:
print('Finding missing values in training data')
missing_data=data.isnull().sum()
print(missing_data[missing_data>0])
print('finding missing values in testing data')
missing_test=test_data.isnull().sum()
print(missing_test[missing_test>0])

Finding missing values in training data
RoadType        600
Temperature    2495
Weather         797
dtype: int64
finding missing values in testing data
RoadType        324
Temperature    1349
Weather         431
dtype: int64


In [57]:
#now let us check for the column types before fixing the missing data
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 non-null  str    
 2   day            77299 non-null  int64  
 3   timestamp      77299 non-null  str    
 4   demand         77299 non-null  float64
 5   RoadType       76699 non-null  str    
 6   NumberofLanes  77299 non-null  int64  
 7   LargeVehicles  77299 non-null  str    
 8   Landmarks      77299 non-null  str    
 9   Temperature    74804 non-null  float64
 10  Weather        76502 non-null  str    
dtypes: float64(2), int64(3), str(6)
memory usage: 9.3 MB


So now, we just have to fill in the missing values(mode for weather and road whereas median for temperature)

In [58]:

weather_mode = data['Weather'].mode()[0]
road_mode = data['RoadType'].mode()[0]
temp_median = data['Temperature'].median()

print(f'Weather mode: {weather_mode}')
print(f'RoadType mode: {road_mode}')
print(f'Temperature median: {temp_median}')

data['Weather'] = data['Weather'].fillna(weather_mode)
data['RoadType'] = data['RoadType'].fillna(road_mode)
data['Temperature'] = data['Temperature'].fillna(temp_median)

test_data['Weather'] = test_data['Weather'].fillna(weather_mode)
test_data['RoadType'] = test_data['RoadType'].fillna(road_mode)
test_data['Temperature'] = test_data['Temperature'].fillna(temp_median)

print("\n=== Missing values filled and saved successfully! ===")

Weather mode: Sunny
RoadType mode: Residential
Temperature median: 16.382587216677827

=== Missing values filled and saved successfully! ===


Now, we have to convert text columns to number encoded values by using pandas encoder

In [59]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = data[col].astype(str)
    test_data[col] = test_data[col].astype(str)
    data[col] = le.fit_transform(data[col])
    test_data[col] = test_data[col].map(lambda s: s if s in le.classes_ else le.classes_[0])
    test_data[col] = le.transform(test_data[col])
print('Data encoded succesfully')

Data encoded succesfully


feature engineering:

In [ ]:
data['hour']=data['timestamp'].str.split(':').str[0].astype(int)
data['minute']=data['timestamp'].str.split(':').str[1].astype(int)
test_data['hour']=test_data['timestamp'].str.split(':').str[0].astype(int)
test_data['minute']=test_data['timestamp'].str.split(':').str[1].astype(int)

In [ ]:
from xgboost import XGBRegressor

features=['geohash','day','hour','minute','RoadType','NumberofLanes','LargeVehicles','Landmarks','Temperature','Weather']

X=data[features]
y=data['demand']
X_test=test_data[features]

model=XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
model.fit(X, y)

preds=model.predict(X_test)
submission=pd.DataFrame({'Index':test_data['Index'],'demand':preds})
submission.to_csv('submission.csv', index=False)
print(submission.head())

   Index    demand
0      0  0.037902
1      1  0.025239
2      2  0.030684
3      3  0.030744
4      4  0.047193


In [ ]:
from sklearn.metrics import r2_score

train_preds=model.predict(X)
score=r2_score(y, train_preds) * 100
print(f'R2 score:{score:.2f}')

R2 score: 87.54


In [63]:
print(submission.shape)
print(submission.columns.tolist())
print(submission.head())

(41778, 2)
['Index', 'demand']
   Index    demand
0      0  0.037902
1      1  0.025239
2      2  0.030684
3      3  0.030744
4      4  0.047193
